In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# 1. Define the IBGE SIDRA API URL (Table 6579 - Municipalities - Latest year)
url = "https://apisidra.ibge.gov.br/values/t/6579/n6/all/p/last%201"

print("Starting IBGE API data extraction...")
response = requests.get(url)

# 2. HTTP Request and Validation
if response.status_code == 200:
    # Convert response to a list of dictionaries (JSON)
    json_data = response.json()
    
    # 3. Create DataFrame bypassing SparkContext
    # We use Pandas to parse the JSON list, then convert directly to a PySpark DataFrame
    pdf_raw = pd.DataFrame(json_data)
    df_raw = spark.createDataFrame(pdf_raw)
    
    # 4. Clean the First Row (SIDRA API Header)
    df_filtered = df_raw.filter(df_raw["D1C"] != "Nível Territorial (Código)")
    
    # 5. Add Audit Metadata Columns
    df_bronze = df_filtered.withColumn("ingestion_timestamp", current_timestamp()) \
                           .withColumn("source_system", lit("API_SIDRA_6579"))
    
    # 6. Save as a Managed Delta Table
    table_name_bronze = "bronze_ibge_population"
    
    df_bronze.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable(table_name_bronze)
             
    print(f"Bronze load complete! {df_bronze.count()} records saved to managed table '{table_name_bronze}'.")
    display(df_bronze)

else:
    print(f"Request failed. Error code: {response.status_code}")